# Noise-based RC parameter extraction — physics-informed hybrid estimator

**Study design (revision).** Targets are the effective parameters `R_eff`, `C_eff` (what the PSD physically
encodes); nominal `R`, `C` are derived through the calibrated temperature model when T is known.

* **Axis A – generalisation split:** `random` (interpolation on the R×C×T grid), `heldout` (4 interior R values and
  C ∈ {2.2·10⁻⁷, 2.2·10⁻⁴} F never seen in training), `ood_T` (train T < 350 K, test T ≥ 350 K).
* **Axis B – temperature information:** `true`, Gaussian sensor error σ = 5/10/20 K, `none`
  (replaces v1 Exp. 1/2/3; same targets everywhere, so the scenarios are directly comparable).
* **Axis C – seeds:** 10 for random/true, 6 elsewhere; mean ± std and paired Wilcoxon tests.

**Methods** (identical inputs, split and training budget):

| | |
|---|---|
| M0 | scikit-learn MLP (v1 baseline) |
| M1 | plain PyTorch MLP (ReLU, no LayerNorm/GELU/dropout) on the physics features – fair MLP baseline |
| M2 | Twin network (two independent v1-architecture branches for R and C), data loss only |
| M3 | M2 + spectral physics loss during training – physics-loss (PINN) ablation |
| M4 | classical system identification: per-sample nonlinear least-squares fit of the calibrated forward model |
| **M5** | **hybrid: M1 prediction as prior + per-sample MAP refinement with the forward model at inference** |
| M5-M2 | the same refinement on the M2 backbone (backbone-independence check) |
| M2-raw | M2 on the raw log-PSD only (feature ablation; random/true only) |

**Forward model** (bin-averaged one-pole PSD, calibrated gain κ on the training split):
$\bar S_k = \kappa\,\frac{2kT}{\pi C\,\Delta f}\left[\arctan(2\pi RC f_{k}^{hi}) - \arctan(2\pi RC f_{k}^{lo})\right]$,
used on 1.2–100 kHz and only where the measured roll-off passes the validity gate (drop ≥ 0.8 dex between ~1.5 and 50 kHz;
for τ close to the simulator step
Ts = 1 µs, i.e. mostly 1 nF, the simulated PSD departs from the continuous model).

Simulink settings: Ts = 1 µs, R tolerance 1 %, C tolerance 5 %.
Results are appended per (split, scenario, seed) block to `v3_results.csv`; re-running resumes.

**Data:** `RC_dataset_python.mat` is read from `../data/` (or `data/`) when run locally, or from `/kaggle/input` on Kaggle.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 0.  IMPORTS & CONFIG
# ─────────────────────────────────────────────────────────────────────────────
import os, io, time, glob, contextlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.io as sio
from scipy.stats import wilcoxon

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
from sklearn.neural_network import MLPRegressor

import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
plt.rcParams.update({
    'figure.dpi': 300, 'savefig.dpi': 300, 'font.size': 14, 'axes.titlesize': 18,
    'axes.labelsize': 16, 'axes.labelweight': 'bold', 'xtick.labelsize': 13, 'ytick.labelsize': 13,
    'legend.fontsize': 13, 'figure.titlesize': 20, 'axes.titleweight': 'bold', 'axes.linewidth': 1.2,
    'lines.linewidth': 2.0, 'lines.markersize': 6, 'grid.alpha': 0.35, 'grid.linewidth': 0.8,
    'figure.facecolor': 'white', 'savefig.bbox': 'tight', 'savefig.pad_inches': 0.15,
})

SEED     = 42
# local copy (repository layout) first, then Kaggle inputs (mount points vary)
_cands = ['../data/RC_dataset_python.mat', 'data/RC_dataset_python.mat',
          '/kaggle/input/rc-dataset/RC_dataset_python.mat']
_hits = [p for p in _cands if os.path.exists(p)] or \
        sorted(glob.glob('/kaggle/input/**/RC_dataset_python.mat', recursive=True))
if not _hits:
    raise FileNotFoundError("RC_dataset_python.mat not found: put it in data/ (see data/README.md) "
                            "or attach the dataset via 'Add Input' on Kaggle.")
MAT_PATH = _hits[0]
print('Data   :', MAT_PATH)

K_B    = 1.38e-23
EPS    = 1e-30
N_LOW  = 5                    # low-frequency bins used for the S0 *feature*

# ── Simulink settings (documentation) ────────────────────────────────────
TS_SIM, R_TOL, C_TOL = 1e-6, 0.01, 0.05

# ── forward model / physics ──────────────────────────────────────────────
PHYS_KMIN      = 3            # skip DC bin + Hann main lobe
PHYS_FMAX_HZ   = 1.0e5        # discretisation/aliasing above (see diagnostic)
PHYS_FLOOR     = -28.0        # log10 PSD numerical floor
PHYS_MIN_BINS  = 5
PHYS_GATE_HZ   = 5.0e4        # validity gate: measured drop ~1.5 kHz → 50 kHz ...
PHYS_GATE_DEX  = 0.8          # ... must be ≥ 0.8 dex (≈ f_c ≲ 22 kHz; 0.6 let a few invalid samples through)
PHYS_W         = (0.005, 0.05, 100)   # M3: λ ramp start, end, warm-up epochs
PHYS_DETACH_C  = True         # M3: physics gradient only to the R branch

# ── study design ─────────────────────────────────────────────────────────
TARGETS         = ('R_eff', 'C_eff')
SPLITS          = ['random', 'heldout', 'ood_T']
HELDOUT_R_RANK  = (3, 7, 12, 16)          # ranks in the 20 sorted R values (interior)
HELDOUT_C       = (2.2e-7, 2.2e-4)
OOD_T_MIN       = 350.0
T_SCENARIOS     = {'true': 0.0, 'noise5': 5.0, 'noise10': 10.0, 'noise20': 20.0, 'none': None}
T_RANGE         = (250.0, 400.0)
T_PRIOR_NONE    = (325.0, 150.0 / np.sqrt(12))   # mean / std of the uniform design range
SEEDS_PRIMARY   = list(range(10))         # random split, true T
SEEDS           = list(range(6))          # all other blocks (≥ 6 for a two-sided Wilcoxon p < 0.05)
FEATURE_ABLATION = True
EPOCHS, PATIENCE = 300, 50
REFINE_S_GRID   = [0.0003, 0.001, 0.003, 0.01, 0.03, 0.1]   # likelihood scale for M5, chosen on validation
PRIOR_GRID_FLOOR = True   # M5 prior σ ≥ (log spacing of the training design values)/√12 per parameter
HYBRID_BACKBONES = {'M5': 'M1', 'M5-M2': 'M2'}   # hybrid refinement → backbone network (M5 = primary)
RESULTS_CSV     = 'v3_results.csv'
PRED_DIR        = 'preds'
RUN_STUDY       = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {device}")
if torch.cuda.is_available():
    print(f"  GPU  : {torch.cuda.get_device_name(0)}")


def t64(a):
    return torch.as_tensor(np.asarray(a, dtype=np.float64), dtype=torch.float64, device=device)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 1.  DATA
# ─────────────────────────────────────────────────────────────────────────────
def load_data(mat_path: str, expected_F: int = 1025) -> dict:
    print(f"\n{'='*70}\nDATA LOADING\n{'='*70}")
    mat = sio.loadmat(mat_path, squeeze_me=True, struct_as_record=False)
    def vec(x): return np.asarray(x).squeeze().reshape(-1)

    f_raw   = vec(mat["f"]).astype(np.float64)
    PSD_raw = np.asarray(mat["PSD"]).squeeze()
    if len(f_raw) != expected_F:
        fu = np.unique(f_raw)
        if len(fu) == expected_F: f_raw = fu
        else: raise ValueError(f"freq mismatch: {len(f_raw)}")
    f = f_raw.astype(np.float32)

    if PSD_raw.ndim == 1:
        PSD = PSD_raw.reshape(PSD_raw.size // expected_F, expected_F).astype(np.float32)
    elif PSD_raw.shape[1] == expected_F: PSD = PSD_raw.astype(np.float32)
    elif PSD_raw.shape[0] == expected_F: PSD = PSD_raw.T.astype(np.float32)
    else: raise ValueError(f"PSD shape {PSD_raw.shape}")

    N, F = PSD.shape
    def to_wide(key):
        x = vec(mat[key])
        if x.size == N:    return x.astype(np.float32)
        if x.size == N*F:  return x.reshape(N, F)[:, 0].astype(np.float32)
        raise ValueError(f"{key}: size={x.size}")

    d = dict(R=to_wide("R"), C=to_wide("C"), R_eff=to_wide("R_eff"), C_eff=to_wide("C_eff"),
             T=to_wide("T"), f=f, PSD=PSD, low_idx=np.arange(N_LOW), n_samples=N)
    print(f"  PSD:{PSD.shape}  f: {f[0]:.0f}–{f[-1]:.0f} Hz (Δf = {f[1]-f[0]:.1f} Hz)")
    for k in ('R', 'C', 'T'):
        print(f"  {k}: {len(np.unique(d[k]))} grid values  [{d[k].min():.3g} … {d[k].max():.3g}]")
    print(f"  R_eff: {d['R_eff'].min():.1f}–{d['R_eff'].max():.1f} Ω   "
          f"C_eff: {d['C_eff'].min():.3g}–{d['C_eff'].max():.3g} F")
    return d


def split_random(n, seed=SEED):
    idx = np.arange(n)
    tr, tmp = train_test_split(idx, test_size=0.3, random_state=seed)
    va, te  = train_test_split(tmp, test_size=0.5, random_state=seed)
    return tr, va, te


def make_split(data, kind):
    """Returns (train, val, test) index arrays. Test set is fixed per split (paired comparisons)."""
    if kind == 'random':
        return split_random(data['n_samples'])
    if kind == 'heldout':
        Rv = np.unique(data['R'])
        held_R = Rv[list(HELDOUT_R_RANK)]
        out = (np.isin(data['R'], held_R) |
               np.any([np.isclose(data['C'], c, rtol=1e-3, atol=0) for c in HELDOUT_C], axis=0))
    elif kind == 'ood_T':
        out = data['T'] >= OOD_T_MIN
    else:
        raise ValueError(kind)
    inside, te = np.where(~out)[0], np.where(out)[0]
    tr, va = train_test_split(inside, test_size=0.15, random_state=SEED)
    return tr, va, te

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 2.  INPUT FEATURES  (v1 physics-guided features; mode 'raw' = log-PSD only)
# ─────────────────────────────────────────────────────────────────────────────
def extract_physics_features(PSD_raw, f, low_idx, T_arr=None, mode='basic'):
    """
    mode='basic' (4): log10 S0, log10(S0/4kT) (or log10 S0 without T), log10 f_c estimate, log-log slope
    mode='extended' (8): basic + log10 band means (0–1 k, 1–100 k, 100–500 kHz) + log low/high ratio
    mode='raw' (0)
    """
    N = PSD_raw.shape[0]
    n_feats = {'raw': 0, 'basic': 4, 'extended': 8}[mode]
    feats = np.zeros((N, n_feats), dtype=np.float32)
    if mode == 'raw':
        return feats
    f_pos, pos = f[f > 0], f > 0
    if mode == 'extended':
        bands = [(f >= 0) & (f < 1e3), (f >= 1e3) & (f < 1e5), (f >= 1e5) & (f <= 5e5)]
    for i in range(N):
        psd = PSD_raw[i]
        S0  = float(np.mean(psd[low_idx]) + EPS)
        feats[i, 0] = np.log10(S0)
        feats[i, 1] = (np.log10(S0 / (4.0 * K_B * float(T_arr[i])) + EPS)
                       if T_arr is not None else feats[i, 0])
        psd_pos = psd[pos]
        above = psd_pos > S0 / 2.0
        if above.any() and not above.all():
            fc_est = float(f_pos[min(int(np.where(above)[0][-1]) + 1, len(f_pos) - 1)])
        else:
            fc_est = float(f_pos[len(f_pos) // 2])
        feats[i, 2] = np.log10(fc_est + EPS)
        q  = len(f_pos) * 3 // 4
        lf, lp = np.log10(f_pos[q:] + EPS), np.log10(psd_pos[q:] + EPS)
        slope = float(np.polyfit(lf, lp, 1)[0]) if (len(lf) > 2 and np.ptp(lf) > 0) else -2.0
        feats[i, 3] = float(np.clip(slope, -4.0, 0.0))
        if mode == 'extended':
            m = [np.mean(psd[b]) for b in bands]
            feats[i, 4:7] = np.log10(np.array(m) + EPS)
            feats[i, 7]   = np.log10(m[0] / (m[2] + EPS) + EPS)
    return feats

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 3.  FORWARD MODEL, CALIBRATION, VALIDITY GATE
# ─────────────────────────────────────────────────────────────────────────────
LOG10_2K_PI = float(np.log10(2.0 * K_B / np.pi))


def bin_edges(f):
    f  = np.asarray(f, dtype=np.float64)
    df = float(f[1] - f[0])
    return np.clip(f - df / 2, 0.0, None), f + df / 2, df


def phys_bin_mask(f):
    return (np.arange(len(f)) >= PHYS_KMIN) & (np.asarray(f) <= PHYS_FMAX_HZ)


def gate_bins(f):
    kg = int(np.argmin(np.abs(np.asarray(f) - PHYS_GATE_HZ)))
    return np.arange(PHYS_KMIN, PHYS_KMIN + 3), np.arange(kg - 2, kg + 3)


def phys_gate(log_psd, f):
    """True where the measured roll-off is large enough for the forward model to hold (uses the PSD only)."""
    g_ref, g_hi = gate_bins(f)
    return (log_psd[:, g_ref].mean(axis=1) - log_psd[:, g_hi].mean(axis=1)) >= PHYS_GATE_DEX


def lorentzian_binavg_log10(lR, lC, T, f_lo, f_hi, df, log_kappa=0.0):
    """
    log10 of the bin-averaged one-pole PSD (torch float64).  lR, lC, T: (B,), f_lo, f_hi: (K,)
    S̄_k = 2kT/(π C Δf) · [atan(2πτ f_hi) − atan(2πτ f_lo)],  τ = RC   (→ 4kTR plateau, kT/(π²f²RC²) tail)
    The arctan difference is evaluated as atan((b−a)/(1+ab)) to avoid cancellation for f >> f_c.
    """
    w   = 2.0 * np.pi * 10.0 ** (lR + lC)
    a   = w[:, None] * f_lo[None, :]
    b   = w[:, None] * f_hi[None, :]
    dat = torch.atan((b - a) / (1.0 + a * b))
    return (LOG10_2K_PI - np.log10(df) + torch.log10(T)[:, None]
            - lC[:, None] + torch.log10(dat) + log_kappa)


def fit_tempco(x_int, x_eff, T):
    """Least-squares x_eff/x_int = a + b·T; returns (a, b), R²."""
    T     = np.asarray(T, dtype=np.float64)
    ratio = np.asarray(x_eff, dtype=np.float64) / np.asarray(x_int, dtype=np.float64)
    A     = np.stack([np.ones_like(T), T], axis=1)
    coef, *_ = np.linalg.lstsq(A, ratio, rcond=None)
    res   = ratio - A @ coef
    return (float(coef[0]), float(coef[1])), float(1.0 - (res**2).sum() / ((ratio - ratio.mean())**2).sum())


@torch.no_grad()
def forward_residuals(data, idx, calib, lR, lC, T):
    bm    = calib['bmask']
    logS  = np.log10(data['PSD'][idx][:, bm].astype(np.float64) + EPS)
    model = lorentzian_binavg_log10(t64(lR), t64(lC), t64(T), t64(calib['f_lo']), t64(calib['f_hi']),
                                    calib['df'], calib['log_kappa']).cpu().numpy()
    r = logS - model
    r[logS <= PHYS_FLOOR] = np.nan
    return r


def calibrate_physics(data, tr_idx, verbose=True):
    """Simulator calibration on the TRAINING split only (lab calibration with known T and parts)."""
    f_lo, f_hi, df = bin_edges(data['f'])
    bm = phys_bin_mask(data['f'])
    g_ref, g_hi = gate_bins(data['f'])
    calib = dict(bmask=bm, f_lo=f_lo[bm], f_hi=f_hi[bm], df=df, log_kappa=0.0, g_ref=g_ref, g_hi=g_hi)
    T = data['T'][tr_idx]
    r = forward_residuals(data, tr_idx, calib, np.log10(data['R_eff'][tr_idx]),
                          np.log10(data['C_eff'][tr_idx]), T)
    gate = phys_gate(np.log10(data['PSD'][tr_idx] + EPS), data['f'])
    calib['log_kappa'] = float(np.nanmedian(r[gate]))
    r -= calib['log_kappa']
    # per-bin scatter about the model (Welch estimation noise + residual model error), gated samples
    calib['sigma_bin'] = float(np.nanmedian(np.nanstd(r[gate], axis=1)))
    calib['coefR'], r2R = fit_tempco(data['R'][tr_idx], data['R_eff'][tr_idx], T)
    calib['coefC'], r2C = fit_tempco(data['C'][tr_idx], data['C_eff'][tr_idx], T)
    if verbose:
        print(f"\n{'='*70}\nPHYSICS CALIBRATION (train split, n={len(tr_idx)})\n{'='*70}")
        print(f"  physics bins : {bm.sum()}  ({f_lo[bm][0]:.0f} – {f_hi[bm][-1]:.0f} Hz)")
        print(f"  validity gate: {gate.mean()*100:.1f}% of training samples pass")
        print(f"  log10 κ      : {calib['log_kappa']:+.4f}   per-bin σ = {calib['sigma_bin']:.4f} dex")
        print(f"  R_eff/R = {calib['coefR'][0]:.4f} + {calib['coefR'][1]:.3e}·T   (R² = {r2R:.4f})")
        print(f"  C_eff/C = {calib['coefC'][0]:.4f} + {calib['coefC'][1]:.3e}·T   (R² = {r2C:.4f};"
              f" C tolerance {C_TOL:.0%} dominates)")
    return calib


def physics_diagnostic(data, tr_idx, calib):
    """Legacy v1 target vs new forward model, with true labels (answers why v1's physics loss failed)."""
    T, C = data['T'][tr_idx], data['C'][tr_idx]
    lRe, lCe = np.log10(data['R_eff'][tr_idx]), np.log10(data['C_eff'][tr_idx])
    lp_low = np.log10(data['PSD'][tr_idx][:, data['low_idx']].astype(np.float64) + EPS)
    lp_low = np.where(lp_low > PHYS_FLOOR, lp_low, np.nan)
    legacy = np.nanmean(lp_low, axis=1) - np.log10(4.0 * K_B * T)
    r      = forward_residuals(data, tr_idx, calib, lRe, lCe, T)
    gate   = phys_gate(np.log10(data['PSD'][tr_idx] + EPS), data['f'])
    fc     = 1.0 / (2 * np.pi * data['R_eff'][tr_idx] * data['C_eff'][tr_idx])
    rms    = lambda x: float(np.sqrt(np.nanmean(x ** 2))) if np.isfinite(x).any() else np.nan
    rows = []
    for c in np.unique(C):
        m = np.isclose(C, c, rtol=1e-3, atol=0.0)
        rows.append({'C': f"{c:.1e}", 'n': int(m.sum()), 'median f_c [Hz]': np.median(fc[m]),
                     'v1 target err (dex)': np.nanmedian(legacy[m] - lRe[m]),
                     'model rms (dex)': rms(r[m]), 'gated rms (dex)': rms(r[m & gate]),
                     'gate pass %': 100 * gate[m].mean()})
    tab = pd.DataFrame(rows)
    print(f"\n{'='*70}\nPHYSICS DIAGNOSTIC (true labels)\n{'='*70}")
    print(tab.to_string(index=False, float_format=lambda x: f"{x:9.3f}"))
    print(f"\n  v1 loss E[(target − log10 R)²] = {np.nanmean((legacy - np.log10(data['R'][tr_idx]))**2):.2f} dex²"
          f"   new model, gated: {np.nanmean(r[gate]**2):.5f} dex²")
    tab.to_csv('physics_diagnostic.csv', index=False)

    fmid = 0.5 * (calib['f_lo'] + calib['f_hi'])
    fig, ax = plt.subplots(figsize=(10, 6))
    for c in np.unique(C):
        m = np.isclose(C, c, rtol=1e-3, atol=0.0)
        ax.semilogx(fmid, np.nanmedian(r[m], axis=0), lw=1.8, label=f"C={c:.1e}")
    ax.axhline(0, color='k', ls='--')
    ax.set_xlabel('Frequency (Hz)'); ax.set_ylabel('log10 S_meas − log10 S_model')
    ax.set_title('Forward-model residual (true labels)'); ax.legend(fontsize=10, ncol=2)
    plt.tight_layout(); plt.savefig('physics_diagnostic.png', dpi=200); plt.show()
    return tab

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 4.  NETWORKS
# ─────────────────────────────────────────────────────────────────────────────
class Branch(nn.Module):
    """v1 architecture (LayerNorm + GELU + dropout) or plain (ReLU only) for the M1 ablation."""
    def __init__(self, in_dim, hidden=(256, 512, 512, 256), dropout=0.15, plain=False):
        super().__init__()
        layers, prev = [], in_dim
        for h in hidden:
            layers += ([nn.Linear(prev, h), nn.ReLU()] if plain else
                       [nn.Linear(prev, h), nn.LayerNorm(h), nn.GELU(), nn.Dropout(dropout)])
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


class TwinNet(nn.Module):
    """Two independent branches for log R and log C (no shared weights)."""
    def __init__(self, in_dim, plain=False):
        super().__init__()
        self.branches = nn.ModuleList([Branch(in_dim, plain=plain) for _ in range(2)])

    def forward(self, x):
        return torch.cat([b(x) for b in self.branches], dim=1)


# ─────────────────────────────────────────────────────────────────────────────
# 5.  PHYSICS LOSS (M3)
# ─────────────────────────────────────────────────────────────────────────────
def build_physics_loss(calib, slog, use_level):
    """
    Spectral forward-model loss on predicted (R_eff, C_eff), gated samples only.
    use_level=True : E_k[r_k²]          (needs T)
    use_level=False: E_k[(r_k − r̄)²]    (T-free shape term, constrains τ = RC)
    """
    bins  = torch.tensor(np.where(calib['bmask'])[0], dtype=torch.long, device=device)
    g_ref = torch.tensor(calib['g_ref'], dtype=torch.long, device=device)
    g_hi  = torch.tensor(calib['g_hi'],  dtype=torch.long, device=device)
    f_lo, f_hi = t64(calib['f_lo']), t64(calib['f_hi'])
    m, s  = t64(slog.mean_), t64(slog.scale_)

    def loss_fn(log_psd, pred_norm, T_batch):
        lp = log_psd[:, bins].double()
        y  = pred_norm.double() * s + m
        lR, lC = y[:, 0], (y[:, 1].detach() if PHYS_DETACH_C else y[:, 1])
        T  = (T_batch.view(-1).double().clamp(min=1.0) if T_batch is not None
              else torch.full_like(lR, T_PRIOR_NONE[0]))
        model = lorentzian_binavg_log10(lR, lC, T, f_lo, f_hi, calib['df'], calib['log_kappa'])
        valid = (lp > PHYS_FLOOR).double()
        n     = valid.sum(dim=1)
        drop  = log_psd[:, g_ref].mean(dim=1) - log_psd[:, g_hi].mean(dim=1)
        ok    = (n >= PHYS_MIN_BINS) & (drop >= PHYS_GATE_DEX)
        if not ok.any():
            return torch.tensor(0.0, device=device)
        r = (lp - model) * valid
        if not use_level:
            r = (r - (r.sum(dim=1) / n.clamp(min=1.0))[:, None]) * valid
        return ((r ** 2).sum(dim=1) / n.clamp(min=1.0))[ok].mean().float()
    return loss_fn


def no_physics_loss(log_psd, pred_norm, T_batch):
    return torch.tensor(0.0, device=device)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 6.  DATASET BUILDER
# ─────────────────────────────────────────────────────────────────────────────
def make_loaders(data, split, T_feat=None, mode='basic', batch=64):
    tr, va, te = split
    log_Y   = np.log10(np.stack([data[k] for k in TARGETS], axis=1) + EPS).astype(np.float32)
    log_PSD = np.log10(data['PSD'] + EPS).astype(np.float32)
    feats   = extract_physics_features(data['PSD'], data['f'], data['low_idx'], T_arr=T_feat, mode=mode)
    X_full  = np.hstack([log_PSD, feats]).astype(np.float32)
    F_raw   = log_PSD.shape[1]
    sx, slog = StandardScaler().fit(X_full[tr]), StandardScaler().fit(log_Y[tr])

    def make_ds(idx):
        ts = [torch.tensor(sx.transform(X_full[idx]).astype(np.float32)),
              torch.tensor(X_full[idx, :F_raw], dtype=torch.float32),
              torch.tensor(slog.transform(log_Y[idx]).astype(np.float32))]
        if T_feat is not None:
            ts.append(torch.tensor(T_feat[idx], dtype=torch.float32))
        return TensorDataset(*ts)

    return dict(tr_ld=DataLoader(make_ds(tr), batch_size=batch, shuffle=True, drop_last=True),
                v_ld=DataLoader(make_ds(va), batch_size=batch, shuffle=False),
                Xtr_s=sx.transform(X_full[tr]).astype(np.float32),
                ytr_s=slog.transform(log_Y[tr]).astype(np.float32),
                Xva_s=sx.transform(X_full[va]).astype(np.float32),
                Xte_s=sx.transform(X_full[te]).astype(np.float32),
                yva_log=log_Y[va], yte_log=log_Y[te], slog=slog, n_in=X_full.shape[1])

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 7.  TRAINER  (per-branch AdamW / ReduceLROnPlateau / early stopping on the branch's own val data loss)
# ─────────────────────────────────────────────────────────────────────────────
class Trainer:
    def __init__(self, model, physics_loss_fn, lr=1e-3, phys_w=(0.0, 0.0, 0),
                 sched_cfg=((0.8, 25), (0.7, 20))):
        self.model   = model.to(device)
        self.phys_fn = physics_loss_fn
        self.opts    = [optim.AdamW(b.parameters(), lr=lr, weight_decay=1e-5) for b in model.branches]
        self.scheds  = [optim.lr_scheduler.ReduceLROnPlateau(o, mode='min', factor=fac, patience=pat,
                                                             min_lr=1e-6)
                        for o, (fac, pat) in zip(self.opts, sched_cfg)]
        self.pw_start, self.pw_end, self.warmup = phys_w
        self.hist   = dict(train=[], val_R=[], val_C=[], phys=[], phys_w=[])
        self.epoch  = 0
        self.active = [True, True]

    def _pw(self):
        if self.pw_end == 0.0:
            return 0.0
        if self.warmup == 0 or self.epoch >= self.warmup:
            return self.pw_end
        return self.pw_start + (self.epoch / self.warmup) * (self.pw_end - self.pw_start)

    def _step(self, batch, use_T):
        if use_T:
            X_s, log_psd, y_s, T_raw = [b.to(device) for b in batch]
            T_batch = T_raw.view(-1, 1)
        else:
            X_s, log_psd, y_s = [b.to(device) for b in batch]
            T_batch = None
        pred   = self.model(X_s)
        d_cols = ((pred - y_s) ** 2).mean(dim=0)
        p_loss = self.phys_fn(log_psd, pred, T_batch)
        return d_cols.sum() + self._pw() * p_loss, p_loss, d_cols

    def fit(self, tr_ld, v_ld, use_T, epochs=EPOCHS, patience=PATIENCE):
        best, wait, best_state = [1e18, 1e18], [0, 0], [None, None]
        t0 = time.time()
        for ep in range(epochs):
            self.epoch = ep
            self.model.train()
            tl = tp = 0.0
            for batch in tr_ld:
                for o in self.opts: o.zero_grad()
                loss, p, _ = self._step(batch, use_T)
                loss.backward()
                for j, (b, o) in enumerate(zip(self.model.branches, self.opts)):
                    if self.active[j]:
                        nn.utils.clip_grad_norm_(b.parameters(), 1.0)
                        o.step()
                tl += loss.item(); tp += p.item()
            self.model.eval()
            vd = np.zeros(2)
            with torch.no_grad():
                for batch in v_ld:
                    vd += self._step(batch, use_T)[2].cpu().numpy()
            vd /= len(v_ld)
            for k, v in zip(('train', 'val_R', 'val_C', 'phys', 'phys_w'),
                            (tl / len(tr_ld), vd[0], vd[1], tp / len(tr_ld), self._pw())):
                self.hist[k].append(v)
            for j in range(2):
                if not self.active[j]:
                    continue
                self.scheds[j].step(vd[j])
                if vd[j] < best[j]:
                    best[j], wait[j] = vd[j], 0
                    best_state[j] = {k: v.clone() for k, v in self.model.branches[j].state_dict().items()}
                else:
                    wait[j] += 1
                    self.active[j] = wait[j] < patience
            if not any(self.active):
                break
        for j in range(2):
            self.model.branches[j].load_state_dict(best_state[j])
        self.train_time = time.time() - t0
        self.epochs_run = ep + 1

    @torch.no_grad()
    def predict(self, X_scaled_np):
        self.model.eval()
        return self.model(torch.tensor(X_scaled_np, dtype=torch.float32, device=device)).cpu().numpy()


def train_net(L, use_T, seed, plain=False, phys_fn=None):
    """Returns (trainer, log10 predictions on val, on test, inference seconds on test)."""
    torch.manual_seed(seed); np.random.seed(seed)
    tr = Trainer(TwinNet(L['n_in'], plain=plain), phys_fn or no_physics_loss,
                 phys_w=PHYS_W if phys_fn is not None else (0.0, 0.0, 0))
    tr.fit(L['tr_ld'], L['v_ld'], use_T)
    pv = L['slog'].inverse_transform(tr.predict(L['Xva_s']))
    t0 = time.time()
    pt = L['slog'].inverse_transform(tr.predict(L['Xte_s']))
    return tr, pv, pt, time.time() - t0


def run_sklearn_mlp(L, seed):
    mlp = MLPRegressor(hidden_layer_sizes=(256, 512, 256), activation='relu', solver='adam',
                       max_iter=500, random_state=seed, early_stopping=True, validation_fraction=0.1)
    t0 = time.time()
    mlp.fit(L['Xtr_s'], L['ytr_s'])
    tt = time.time() - t0
    t0 = time.time()
    pt = L['slog'].inverse_transform(mlp.predict(L['Xte_s']).reshape(-1, 2))
    n_par = sum(w.size for w in mlp.coefs_) + sum(b.size for b in mlp.intercepts_)
    return pt, tt, time.time() - t0, n_par

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 8.  PER-SAMPLE INVERSION: batched Levenberg–Marquardt on the forward model
#     M4 (NLS): flat prior, grid-search initialisation
#     M5 (hybrid MAP): network prediction as Gaussian prior on (log R_eff, log C_eff),
#                      T as a latent with prior N(T_in, σ_T²) when the sensor is noisy or absent
#     J(θ) = (1/n) Σ_k r_k² / s²  +  Σ ((θ − μ)/σ)²  +  ((T − T_in)/σ_T)²
# ─────────────────────────────────────────────────────────────────────────────
def _make_ctx(logpsd, T_in, calib, s_lik, mu=None, sig=None, sigT=0.0):
    bm = calib['bmask']
    lp = t64(logpsd[:, bm])
    valid = (lp > PHYS_FLOOR).double()
    return dict(lp=lp, valid=valid, sqn=valid.sum(1).clamp(min=1.0).sqrt(),
                lo=t64(calib['f_lo']), hi=t64(calib['f_hi']), df=calib['df'], kappa=calib['log_kappa'],
                T0=t64(T_in), sigT=float(sigT), fit_T=sigT > 0, s=float(s_lik),
                mu=None if mu is None else t64(mu), sig=None if sig is None else t64(sig))


def _residuals(th, c):
    lR = th[:, 0].clamp(0.5, 5.0)
    lC = th[:, 1].clamp(-11.0, -1.0)
    T  = (c['T0'] + c['sigT'] * th[:, 2]).clamp(150.0, 500.0) if c['fit_T'] else c['T0']
    m  = lorentzian_binavg_log10(lR, lC, T, c['lo'], c['hi'], c['df'], c['kappa'])
    parts = [(c['lp'] - m) * c['valid'] / c['s'] / c['sqn'][:, None]]
    if c['mu'] is not None:
        parts.append((th[:, :2] - c['mu']) / c['sig'])
    if c['fit_T']:
        parts.append(th[:, 2:3])
    return torch.cat(parts, dim=1)


@torch.no_grad()
def lm_solve(theta0, c, iters=40):
    th   = theta0.clone()
    P    = th.shape[1]
    r    = _residuals(th, c)
    cost = (r ** 2).sum(1)
    lam  = torch.full_like(cost, 1e-2)
    eye  = torch.eye(P, dtype=th.dtype, device=th.device)
    h    = [1e-6, 1e-6, 1e-4][:P]
    for _ in range(iters):
        J = torch.stack([(_residuals(th + h[j] * eye[j], c) - _residuals(th - h[j] * eye[j], c)) / (2 * h[j])
                         for j in range(P)], dim=2)
        A = J.transpose(1, 2) @ J
        g = (J.transpose(1, 2) @ r[:, :, None])[:, :, 0]
        D = torch.diag_embed(torch.diagonal(A, dim1=1, dim2=2)) + 1e-12 * eye
        step  = -torch.linalg.solve(A + lam[:, None, None] * D, g[:, :, None])[:, :, 0]
        r_new = _residuals(th + step, c)
        c_new = (r_new ** 2).sum(1)
        ok    = c_new < cost
        th    = torch.where(ok[:, None], th + step, th)
        r     = torch.where(ok[:, None], r_new, r)
        cost  = torch.where(ok, c_new, cost)
        lam   = torch.where(ok, lam * 0.3, lam * 10.0).clamp(1e-9, 1e9)
    return th


@torch.no_grad()
def grid_init(c, n_R=24, n_C=57):
    """Coarse grid search over (log R_eff, log C_eff) for the NLS baseline (T fixed at T_in)."""
    gR = torch.linspace(1.5, 3.8, n_R, dtype=torch.float64, device=device)
    gC = torch.linspace(-9.5, -2.5, n_C, dtype=torch.float64, device=device)
    N  = c['lp'].shape[0]
    best_cost = torch.full((N,), float('inf'), dtype=torch.float64, device=device)
    best = torch.zeros((N, 2), dtype=torch.float64, device=device)
    for lR in gR:
        for lC in gC:
            th = torch.stack([torch.full((N,), float(lR), dtype=torch.float64, device=device),
                              torch.full((N,), float(lC), dtype=torch.float64, device=device)], 1)
            cost = (_residuals(th, c) ** 2).sum(1)
            upd  = cost < best_cost
            best_cost = torch.where(upd, cost, best_cost)
            best = torch.where(upd[:, None], th, best)
    return best


def _clip_theta(th):
    return np.stack([np.clip(th[:, 0], 0.5, 5.0), np.clip(th[:, 1], -11.0, -1.0)], axis=1)


def nls_estimate(logpsd, T_in, calib):
    c  = _make_ctx(logpsd, T_in, calib, s_lik=1.0)
    th = lm_solve(grid_init(c), c)
    return _clip_theta(th.cpu().numpy())


def hybrid_estimate(logpsd, T_in, calib, prior_mu, prior_sig, sigT, s_lik, gate):
    """MAP refinement on gated samples; others keep the network prediction."""
    out = prior_mu.copy()
    if gate.any():
        c   = _make_ctx(logpsd[gate], T_in[gate], calib, s_lik, mu=prior_mu[gate], sig=prior_sig, sigT=sigT)
        th0 = t64(np.hstack([prior_mu[gate], np.zeros((gate.sum(), 1))]) if sigT > 0 else prior_mu[gate])
        out[gate] = _clip_theta(lm_solve(th0, c).cpu().numpy()[:, :2])
    return out

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 9.  CRAMÉR–RAO BOUND (in-band data, calibrated model, per-bin σ from the training residuals)
# ─────────────────────────────────────────────────────────────────────────────
@torch.no_grad()
def crlb_log_sigma(data, idx, calib, sigT):
    """σ_CRLB of (log10 R_eff, log10 C_eff) per sample; T known if sigT == 0, else nuisance with prior σ_T."""
    logpsd = np.log10(data['PSD'][idx].astype(np.float64) + EPS)
    T  = data['T'][idx].astype(np.float64)
    c  = _make_ctx(logpsd, T, calib, s_lik=calib['sigma_bin'], sigT=sigT)
    c['sqn'] = torch.ones_like(c['sqn'])            # true likelihood (no 1/n tempering)
    th = t64(np.stack([np.log10(data['R_eff'][idx]), np.log10(data['C_eff'][idx])], 1))
    if sigT > 0:
        th = torch.cat([th, torch.zeros_like(th[:, :1])], 1)
    P, h = th.shape[1], [1e-6, 1e-6, 1e-4]
    eye = torch.eye(P, dtype=th.dtype, device=th.device)
    J = torch.stack([(_residuals(th + h[j] * eye[j], c) - _residuals(th - h[j] * eye[j], c)) / (2 * h[j])
                     for j in range(P)], dim=2)
    F = J.transpose(1, 2) @ J                       # prior on T already enters as a residual row
    # In-band data that only see the f⁻² tail constrain R·C² alone: F is (numerically) singular and the
    # bound is infinite — mark those samples as non-identifiable instead of letting a pseudo-inverse hide it.
    singular = torch.linalg.cond(F) > 1e10
    eye = torch.eye(P, dtype=F.dtype, device=F.device)
    cov = torch.linalg.solve(torch.where(singular[:, None, None], eye, F), eye.expand_as(F))
    sig = torch.sqrt(torch.diagonal(cov, dim1=1, dim2=2)[:, :2]).cpu().numpy()
    sig[singular.cpu().numpy()] = np.inf
    return sig


def dex_to_mape(sigma_dex):
    """Expected |relative error| (%) for a Gaussian error of σ decades in log10."""
    return 100.0 * np.log(10.0) * sigma_dex * np.sqrt(2.0 / np.pi)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 10. METRICS
# ─────────────────────────────────────────────────────────────────────────────
def metrics(true_phys, pred_phys):
    """MAPE (%), R², log-R² and mean absolute log10 error (dex)."""
    lt, lp = np.log10(true_phys + EPS), np.log10(pred_phys + EPS)
    return dict(MAPE=float(np.mean(np.abs(pred_phys - true_phys) / true_phys) * 100),
                R2=float(r2_score(true_phys, pred_phys)),
                log_R2=float(r2_score(lt, lp)),
                log_MAE_dex=float(np.mean(np.abs(lp - lt))))

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 11. ONE BLOCK = (split, temperature scenario, seed): all methods on identical data
# ─────────────────────────────────────────────────────────────────────────────
def temperature_input(data, scen, seed):
    """Returns (T fed to features/physics or None, T used by the inversion, σ_T of its prior)."""
    sigma = T_SCENARIOS[scen]
    T_true = data['T'].astype(np.float64)
    if sigma is None:
        return None, np.full_like(T_true, T_PRIOR_NONE[0]), T_PRIOR_NONE[1]
    rng = np.random.default_rng([seed, 7])
    T_in = T_true + (rng.normal(0.0, sigma, T_true.shape) if sigma > 0 else 0.0)
    return T_in.astype(np.float32), T_in, sigma


def design_floor(data, tr):
    """Uniform-quantisation width of the training design in log10, per parameter.

    A network trained on a discrete design cannot be trusted below the spacing of
    that design (it tends to snap to the nearest trained value), whatever its
    in-distribution validation error says. The M5 prior is therefore never
    narrower than spacing/√12. Uses training labels only.
    """
    out = []
    for key in ('R', 'C'):
        u = np.unique(np.round(np.log10(data[key][tr].astype(np.float64)), 6))
        out.append(float(np.median(np.diff(u))) / np.sqrt(12) if len(u) > 1 else 0.0)
    return np.array(out)


def run_block(data, split_name, split, calib, scen, seed):
    tr, va, te = split
    T_feat, T_inv, sigT = temperature_input(data, scen, seed)
    use_T  = T_feat is not None
    L      = make_loaders(data, split, T_feat=T_feat, mode='basic')
    true_l = L['yte_log']
    logpsd = np.log10(data['PSD'].astype(np.float64) + EPS)
    gate_te = phys_gate(logpsd[te], data['f'])
    P, info, PV, S_LIK, SIG = {}, {}, {}, {}, {}

    pt, tt, it, npar = run_sklearn_mlp(L, seed)
    P['M0'], info['M0'] = pt, (tt, it, npar)

    for name, plain, phys in (('M1', True, False), ('M2', False, False), ('M3', False, True)):
        fn = build_physics_loss(calib, L['slog'], use_level=use_T) if phys else None
        trn, pv, pt, it = train_net(L, use_T, seed, plain=plain, phys_fn=fn)
        P[name] = pt
        info[name] = (trn.train_time, it, sum(p.numel() for p in trn.model.parameters()))
        if name in HYBRID_BACKBONES.values():
            PV[name] = pv

    t0 = time.time()
    P['M4'] = nls_estimate(logpsd[te], T_inv[te], calib)
    info['M4'] = (0.0, time.time() - t0, 2)

    # M5 (on M1, primary) and M5-M2: physics-based MAP refinement of a backbone network.
    # Prior width from the backbone's validation error (floored by the design spacing);
    # likelihood scale s chosen on validation.
    gate_va = phys_gate(logpsd[va], data['f'])
    sT = sigT if scen != 'true' else 0.0
    for hname, bb in HYBRID_BACKBONES.items():
        sig = np.sqrt(np.mean((PV[bb] - L['yva_log']) ** 2, axis=0))
        if PRIOR_GRID_FLOOR:
            sig = np.maximum(sig, design_floor(data, tr))
        best_s, best_err = None, np.inf
        for s in REFINE_S_GRID:
            err = np.abs(hybrid_estimate(logpsd[va], T_inv[va], calib, PV[bb], sig, sT, s, gate_va)
                         - L['yva_log'])
            e = np.mean(err if use_T else err[:, 0])      # C is not refined without T (below)
            if e < best_err:
                best_s, best_err = s, e
        t0 = time.time()
        P[hname] = hybrid_estimate(logpsd[te], T_inv[te], calib, P[bb], sig, sT, best_s, gate_te)
        if not use_T:
            # Without a temperature measurement the spectrum fixes f_c and T·R only, so C ∝ T is
            # not identifiable (see the 'none' CRLB): the refinement cannot add information on C.
            P[hname][:, 1] = P[bb][:, 1]
        info[hname] = (info[bb][0], info[bb][1] + time.time() - t0, info[bb][2])
        S_LIK[hname], SIG[hname] = best_s, sig

    if FEATURE_ABLATION and split_name == 'random' and scen == 'true':
        L_raw = make_loaders(data, split, T_feat=T_feat, mode='raw')
        trn, _, pt, it = train_net(L_raw, use_T, seed)
        P['M2-raw'] = pt
        info['M2-raw'] = (trn.train_time, it, sum(p.numel() for p in trn.model.parameters()))

    rows = []
    aR, bR = calib['coefR']; aC, bC = calib['coefC']
    for m, pl in P.items():
        for j, key in enumerate(TARGETS):
            rows.append(dict(split=split_name, scenario=scen, seed=seed, method=m, param=key,
                             train_s=info[m][0], infer_s=info[m][1], n_params=info[m][2],
                             s_lik=S_LIK.get(m, np.nan),
                             **metrics(10 ** true_l[:, j], 10 ** pl[:, j])))
        if scen == 'true':     # nominal values through the calibrated T-model (T known)
            T = data['T'][te]
            for key, est, ref in (('R', 10 ** pl[:, 0] / (aR + bR * T), data['R'][te]),
                                  ('C', 10 ** pl[:, 1] / (aC + bC * T), data['C'][te])):
                rows.append(dict(split=split_name, scenario=scen, seed=seed, method=m, param=key,
                                 train_s=np.nan, infer_s=np.nan, n_params=np.nan, s_lik=np.nan,
                                 **metrics(ref, est)))
    os.makedirs(PRED_DIR, exist_ok=True)
    np.savez_compressed(f"{PRED_DIR}/{split_name}_{scen}_{seed}.npz", te=te, true=true_l,
                        gate=gate_te, va=va,
                        **{f'{bb}_val': PV[bb] for bb in PV},
                        **{f"prior_sig_{h.replace('-', '_')}": SIG[h] for h in SIG},
                        **{f"s_lik_{h.replace('-', '_')}": S_LIK[h] for h in S_LIK},
                        **{k.replace('-', '_'): v for k, v in P.items()})
    return rows


def run_study(data):
    done = set()
    if os.path.exists(RESULTS_CSV):
        prev = pd.read_csv(RESULTS_CSV, dtype={'scenario': str, 'split': str})
        done = set(zip(prev.split, prev.scenario, prev.seed))
    for split_name in SPLITS:
        split = make_split(data, split_name)
        calib = calibrate_physics(data, split[0], verbose=True)
        print(f"  [{split_name}] train={len(split[0])} val={len(split[1])} test={len(split[2])}")
        for scen in T_SCENARIOS:
            for seed in (SEEDS_PRIMARY if (split_name == 'random' and scen == 'true') else SEEDS):
                if (split_name, scen, seed) in done:
                    continue
                t0 = time.time()
                rows = run_block(data, split_name, split, calib, scen, seed)
                pd.DataFrame(rows).to_csv(RESULTS_CSV, mode='a', index=False,
                                          header=not os.path.exists(RESULTS_CSV))
                r = {(x['method'], x['param']): x['MAPE'] for x in rows}
                print(f"  {split_name:7s} {scen:7s} seed={seed}  R_eff MAPE: " +
                      '  '.join(f"{m}={r[(m, 'R_eff')]:.1f}" for m in ('M0', 'M1', 'M2', 'M3', 'M4', 'M5', 'M5-M2')) +
                      "  | C_eff: " +
                      '  '.join(f"{m}={r[(m, 'C_eff')]:.1f}" for m in ('M1', 'M2', 'M5', 'M5-M2')) +
                      f"   ({time.time()-t0:.0f} s)")
    return pd.read_csv(RESULTS_CSV, dtype={'scenario': str, 'split': str})

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 12. LOAD, CALIBRATE, DIAGNOSE
# ─────────────────────────────────────────────────────────────────────────────
data  = load_data(MAT_PATH)
split0 = make_split(data, 'random')
calib0 = calibrate_physics(data, split0[0])
diag   = physics_diagnostic(data, split0[0], calib0)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 13. RUN THE STUDY  (resumable)
# ─────────────────────────────────────────────────────────────────────────────
if RUN_STUDY:
    results = run_study(data)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 14. ANALYSIS — tables
# ─────────────────────────────────────────────────────────────────────────────
METHOD_ORDER = ['M0', 'M1', 'M2', 'M2-raw', 'M3', 'M4', 'M5-M2', 'M5']
SCEN_ORDER   = list(T_SCENARIOS)


def pm(g, col):
    return f"{g[col].mean():.2f} ± {g[col].std():.2f}"


def table(df, split, scen, cols=('MAPE', 'R2', 'log_R2', 'log_MAE_dex'), params=TARGETS):
    d = df[(df.split == split) & (df.scenario == scen) & df.param.isin(params)]
    rows = []
    for (m, p), g in d.groupby(['method', 'param']):
        rows.append({'method': m, 'param': p, **{c: pm(g, c) for c in cols}, 'n_seeds': len(g)})
    out = pd.DataFrame(rows)
    out['method'] = pd.Categorical(out['method'], METHOD_ORDER, ordered=True)
    return out.sort_values(['param', 'method'])


def paired_tests(df, pairs=(('M5', 'M1'), ('M5-M2', 'M2'), ('M5', 'M5-M2'), ('M5', 'M0'), ('M5', 'M4'),
                            ('M3', 'M2'), ('M1', 'M2'), ('M2-raw', 'M2'), ('M1', 'M0')), metric='MAPE'):
    out = []
    for (sp, sc, prm), g in df[df.param.isin(TARGETS)].groupby(['split', 'scenario', 'param']):
        piv = g.pivot_table(index='seed', columns='method', values=metric)
        for a, b in pairs:
            if a not in piv or b not in piv:
                continue
            d = (piv[a] - piv[b]).dropna()
            p = wilcoxon(d).pvalue if len(d) >= 6 and (d != 0).any() else np.nan
            out.append(dict(split=sp, scenario=sc, param=prm, comparison=f"{a} vs {b}", n=len(d),
                            A=piv[a].mean(), B=piv[b].mean(), diff_pp=d.mean(),
                            rel_change_pct=100 * d.mean() / piv[b].mean(), wins=int((d < 0).sum()), p=p))
    return pd.DataFrame(out)


def _ape(z, m, j):
    return 100 * np.abs(10 ** z[m][:, j] - 10 ** z['true'][:, j]) / 10 ** z['true'][:, j]


FC_REGIMES = (('below band, R·C² only (f_c < 20 Hz)', 0.0, 20.0),
              ('transition (20 Hz – 1.5 kHz)', 20.0, 1.5e3),
              ('in band (1.5 – 50 kHz)', 1.5e3, 5.0e4),
              ('above gate (f_c > 50 kHz)', 5.0e4, np.inf))


def regime_table(split='random', scen='true'):
    """MAPE per corner-frequency regime (true f_c of the test sample), mean ± std over seeds."""
    rows = []
    for fn in sorted(glob.glob(f"{PRED_DIR}/{split}_{scen}_*.npz")):
        z = np.load(fn); seed = int(fn.rsplit('_', 1)[1].split('.')[0])
        fc = 1.0 / (2 * np.pi * 10 ** z['true'][:, 0] * 10 ** z['true'][:, 1])
        for name, a, b in FC_REGIMES:
            k = (fc >= a) & (fc < b)
            if not k.any():
                continue
            for m in METHOD_ORDER:
                if m.replace('-', '_') not in z.files:
                    continue
                for j, key in enumerate(TARGETS):
                    if m == 'M3' and j == 1:
                        continue
                    rows.append(dict(regime=name, n=int(k.sum()), param=key, method=m, seed=seed,
                                     MAPE=float(np.mean(_ape(z, m.replace('-', '_'), j)[k]))))
    if not rows:
        return None
    d = pd.DataFrame(rows)
    t = (d.groupby(['param', 'regime', 'n', 'method'])['MAPE']
           .agg(lambda s: f"{s.mean():.2f} ± {s.std():.2f}").unstack('method')
           .reindex(columns=[m for m in METHOD_ORDER if m in d.method.unique()]))
    return t.reindex([r[0] for r in FC_REGIMES], level='regime')


def heldout_group_table(scen='true'):
    """Held-out split: which held-out factor drives the error (true T)."""
    rows = []
    Rv = np.unique(data['R']); held_R = Rv[list(HELDOUT_R_RANK)]
    for fn in sorted(glob.glob(f"{PRED_DIR}/heldout_{scen}_*.npz")):
        z = np.load(fn); te = z['te']
        hR = np.isin(data['R'][te], held_R)
        hC = np.any([np.isclose(data['C'][te], c, rtol=1e-3, atol=0) for c in HELDOUT_C], axis=0)
        for name, k in (('unseen R, seen C', hR & ~hC), ('seen R, unseen C', ~hR & hC),
                        ('unseen R, unseen C', hR & hC)):
            for m in ('M0', 'M1', 'M2', 'M3', 'M5-M2', 'M5'):
                for j, key in enumerate(TARGETS):
                    if m == 'M3' and j == 1:
                        continue
                    rows.append(dict(group=name, n=int(k.sum()), param=key, method=m,
                                     MAPE=float(np.mean(_ape(z, m.replace('-', '_'), j)[k]))))
    if not rows:
        return None
    d = pd.DataFrame(rows)
    return (d.groupby(['param', 'group', 'n', 'method'])['MAPE']
              .agg(lambda s: f"{s.mean():.2f} ± {s.std():.2f}").unstack('method'))


if os.path.exists(RESULTS_CSV):
    res = pd.read_csv(RESULTS_CSV, dtype={'scenario': str, 'split': str})
    # M3 differs from M2 only in the R branch (physics gradient is detached from C), so its
    # C predictions are identical to M2's by construction: report M3 for R only.
    res = res[~((res.method == 'M3') & res.param.isin(['C_eff', 'C']))]
    pd.set_option('display.width', 200)
    print(f"\n{'='*70}\nTABLE 1 – random split, true T (mean ± std over seeds)\n{'='*70}")
    t1 = table(res, 'random', 'true'); print(t1.to_string(index=False)); t1.to_csv('table1_main.csv', index=False)

    print(f"\n{'='*70}\nTABLE 1b – nominal R, C derived through the T-model (random, true T)\n{'='*70}")
    print(table(res, 'random', 'true', cols=('MAPE', 'R2'), params=('R', 'C')).to_string(index=False))

    print(f"\n{'='*70}\nTABLE 2 – generalisation (true T): MAPE mean ± std\n{'='*70}")
    t2 = (res[(res.scenario == 'true') & res.param.isin(TARGETS)]
          .groupby(['split', 'param', 'method'])['MAPE'].agg(lambda s: f"{s.mean():.2f} ± {s.std():.2f}")
          .unstack('method').reindex(columns=[m for m in METHOD_ORDER if m in res.method.unique()]))
    print(t2.to_string()); t2.to_csv('table2_generalisation.csv')

    print(f"\n{'='*70}\nTABLE 3 – temperature information (random split): R_eff / C_eff MAPE\n{'='*70}")
    t3 = (res[(res.split == 'random') & res.param.isin(TARGETS)]
          .groupby(['param', 'scenario', 'method'])['MAPE'].mean().unstack('method')
          .reindex(SCEN_ORDER, level='scenario'))
    print(t3.round(2).to_string()); t3.to_csv('table3_temperature.csv')

    print(f"\n{'='*70}\nTABLE 4 – cost (random split, true T)\n{'='*70}")
    t4 = (res[(res.split == 'random') & (res.scenario == 'true') & (res.param == TARGETS[0])]
          .groupby('method')[['n_params', 'train_s', 'infer_s']].mean())
    print(t4.round(3).to_string()); t4.to_csv('table4_cost.csv')

    t5 = regime_table()
    if t5 is not None:
        print(f"\n{'='*70}\nTABLE 5 – error by corner-frequency regime (random, true T)\n{'='*70}")
        print(t5.to_string()); t5.to_csv('table5_regimes.csv')
    t6 = heldout_group_table()
    if t6 is not None:
        print(f"\n{'='*70}\nTABLE 6 – held-out split by unseen factor (true T)\n{'='*70}")
        print(t6.to_string()); t6.to_csv('table6_heldout_groups.csv')

    tests = paired_tests(res)
    print(f"\n{'='*70}\nPAIRED WILCOXON TESTS (MAPE; diff < 0 means A better)\n{'='*70}")
    print(tests.to_string(index=False, float_format=lambda x: f"{x:8.3f}"))
    tests.to_csv('paired_tests.csv', index=False)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 15. ANALYSIS — figures
# ─────────────────────────────────────────────────────────────────────────────
COL = {'M0': '#9e9e9e', 'M1': '#8d6e63', 'M2': '#1565C0', 'M2-raw': '#64b5f6', 'M3': '#7b1fa2',
       'M4': '#2e7d32', 'M5-M2': '#f48fb1', 'M5': '#D32F2F'}

if os.path.exists(RESULTS_CSV):
    # (a) error vs temperature information, with the CRLB
    fig, axes = plt.subplots(1, 2, figsize=(18, 6.5))
    _, _, te0 = split0
    crlb_m = {}
    for scen, sigma in T_SCENARIOS.items():
        sT = T_PRIOR_NONE[1] if sigma is None else sigma
        s  = crlb_log_sigma(data, te0, calib0, sT)
        g  = phys_gate(np.log10(data['PSD'][te0] + EPS), data['f']) & np.all(np.isfinite(s), axis=1)
        crlb_m[scen] = dex_to_mape(np.median(s[g], axis=0))      # identifiable, gated samples
    for ax, (j, key) in zip(axes, enumerate(TARGETS)):
        d = res[(res.split == 'random') & (res.param == key)]
        x = np.arange(len(SCEN_ORDER))
        for m in ('M1', 'M2', 'M3', 'M5-M2', 'M5'):   # M4 off-scale (see Table 5); M3 only differs in R
            if m not in d.method.unique():
                continue
            g = d[d.method == m].groupby('scenario')['MAPE'].agg(['mean', 'std']).reindex(SCEN_ORDER)
            ax.errorbar(x, g['mean'], yerr=g['std'], marker='o', capsize=4, color=COL[m], label=m)
        ax.plot(x, [crlb_m[s][j] for s in SCEN_ORDER], 'k--', marker='s',
                label='CRLB, in-band data only, no prior\n(identifiable samples, median)')
        ax.set_xticks(x); ax.set_xticklabels(['true T' if s == 'true' else 'no T' if T_SCENARIOS[s] is None
                                               else f"σ={T_SCENARIOS[s]:g} K" for s in SCEN_ORDER])
        ax.set_yscale('log'); ax.set_ylabel('MAPE (%)'); ax.set_title(f'{key} vs temperature information')
        ax.legend(fontsize=11, loc='upper left')
    plt.tight_layout(); plt.savefig('fig_temperature_information.png', dpi=200); plt.show()

    # (b) identifiability: CRLB vs corner frequency, with empirical M2/M5 errors (random split, seed 0)
    fc = 1.0 / (2 * np.pi * data['R_eff'][te0] * data['C_eff'][te0])
    fig, axes = plt.subplots(1, 2, figsize=(18, 6.5))
    for ax, (j, key) in zip(axes, enumerate(TARGETS)):
        o = np.argsort(fc)
        for scen, sigma, ls in (('true', 0.0, '-'), ('noise10', 10.0, '--'), ('none', None, ':')):
            s = dex_to_mape(crlb_log_sigma(data, te0, calib0, T_PRIOR_NONE[1] if sigma is None else sigma)[:, j])
            ax.loglog(fc[o], pd.Series(s[o]).replace(np.inf, np.nan)
                      .rolling(25, center=True, min_periods=5).median(), 'k', ls=ls, label=f'CRLB (in-band, no prior), {scen}')
            if scen == 'true':
                nonid = ~np.isfinite(s)
                if nonid.any():
                    ax.axvspan(fc[nonid].min(), fc[nonid].max(), color='orange', alpha=0.12,
                               label='in-band data fix R·C² only')
        f0 = f"{PRED_DIR}/random_true_0.npz"
        if os.path.exists(f0):
            z = np.load(f0)
            edges = np.logspace(np.log10(fc.min()), np.log10(fc.max()), 12)
            mid = np.sqrt(edges[1:] * edges[:-1])
            for m in ('M1', 'M5', 'M4'):
                e = 100 * np.abs(10 ** z[m][:, j] - 10 ** z['true'][:, j]) / 10 ** z['true'][:, j]
                ax.loglog(mid, [np.mean(e[(fc >= a) & (fc < b)]) if ((fc >= a) & (fc < b)).any() else np.nan
                                for a, b in zip(edges[:-1], edges[1:])], 'o-', color=COL[m], label=f'{m} (seed 0)')
        ax.axvline(PHYS_KMIN * float(data['f'][1]), color='gray', alpha=0.5)
        ax.set_xlabel('corner frequency f_c (Hz)'); ax.set_ylabel('MAPE (%)')
        ax.set_title(f'{key}: identifiability'); ax.legend(fontsize=10)
    plt.tight_layout(); plt.savefig('fig_identifiability.png', dpi=200); plt.show()

    # (c) generalisation bars (true T)
    fig, axes = plt.subplots(1, 2, figsize=(18, 6.5))
    for ax, key in zip(axes, TARGETS):
        d = res[(res.scenario == 'true') & (res.param == key)]
        ms = [m for m in ('M0', 'M1', 'M2', 'M3', 'M5-M2', 'M5') if m in d.method.unique()]   # M4: Table 5
        w = 0.8 / len(ms)
        for i, m in enumerate(ms):
            g = d[d.method == m].groupby('split')['MAPE'].agg(['mean', 'std']).reindex(SPLITS)
            ax.bar(np.arange(len(SPLITS)) + i * w, g['mean'], w, yerr=g['std'], capsize=3, color=COL[m], label=m)
        ax.set_xticks(np.arange(len(SPLITS)) + 0.4 - w / 2); ax.set_xticklabels(SPLITS)
        ax.set_yscale('log'); ax.set_ylabel('MAPE (%)'); ax.set_title(f'{key}: generalisation (true T)')
        ax.legend(fontsize=11, ncol=len(ms), loc='upper center', bbox_to_anchor=(0.5, -0.08), frameon=False)
    plt.tight_layout(); plt.savefig('fig_generalisation.png', dpi=200); plt.show()

    # (d) prediction scatter M1 vs M5 (random, true T, seed 0)
    f0 = f"{PRED_DIR}/random_true_0.npz"
    if os.path.exists(f0):
        z = np.load(f0)
        fig, axes = plt.subplots(1, 2, figsize=(16, 7))
        for ax, (j, key) in zip(axes, enumerate(TARGETS)):
            for m in ('M1', 'M5'):
                ax.scatter(z['true'][:, j], z[m][:, j], s=22, alpha=0.5, color=COL[m],
                           label=f"{m}  R²(log) = {r2_score(z['true'][:, j], z[m][:, j]):.4f}")
            lim = [z['true'][:, j].min(), z['true'][:, j].max()]
            ax.plot(lim, lim, 'k--'); ax.set_xlabel(f'true log10 {key}'); ax.set_ylabel(f'predicted log10 {key}')
            ax.legend(fontsize=12)
        plt.tight_layout(); plt.savefig('fig_scatter_M1_M5.png', dpi=200); plt.show()